In [ ]:
# This is the template for the submission. You can develop your algorithm in a regular Python script and copy the code here for submission.

# TEAM NAME ON KAGGLE
# "EXAMPLE_GROUP"

# GROUP NUMBER
# "group_XX"

# TEAM MEMBERS (E-MAIL, LEGI, KAGGLE USERNAME):
# "examplestudent1@ethz.ch", "12-345-678", "eXampl3stdNtone" 
# "examplestudent2@ethz.ch", "12-345-679", "xXexamplestudent2Xx"
# "examplestudent3@ethz.ch", "12-345-670", "mhealth_student_98"

In [2]:
import os
from os import listdir
from os.path import isfile, join
import re
import time
from pathlib import Path
from typing import Iterable

import joblib
import numpy as np
import pandas as pd
from scipy import signal
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from dataclasses import dataclass, field
from typing import Mapping, Sequence
from scipy.signal import welch
from sklearn.metrics import f1_score

from scipy.ndimage import median_filter, uniform_filter1d
from lightgbm import LGBMClassifier

from mhealth_activity import Recording

In [ ]:
TRAIN_DIR = Path("/kaggle/input/competitions/26-mham-ex3/data/train")
TEST_DIR = Path("/kaggle/input/competitions/26-mham-ex3/data/test")

# Step Count Section

In [ ]:
from scipy.signal import butter, filtfilt, find_peaks

# DON'T TOUCH the hyperparameters, they are tuned and won't get better.
def estimate_steps_windowed(
    rec,
    window_s=10.0,
    low_hz=0.7,
    high_hz=3.0,
    peak_prominence=0.14,
    max_step_hz=3.0,
    min_peak_rate_hz=1.3,
    min_std_threshold=0.08,
    min_final_steps_to_keep=80,
):
    ax = rec.data["ax"].values.astype(float)
    ay = rec.data["ay"].values.astype(float)
    az = rec.data["az"].values.astype(float)
    fs = float(rec.data["ax"].samplerate)

    mag = np.sqrt(ax**2 + ay**2 + az**2)
    mag_centered = mag - np.mean(mag)

    nyq = fs / 2.0
    b, a = butter(3, [low_hz / nyq, high_hz / nyq], btype="band")
    filt = filtfilt(b, a, mag_centered)

    win_len = int(window_s * fs)
    min_distance = int(fs / max_step_hz)

    total_steps = 0
    kept_windows = []

    for start in range(0, len(filt), win_len):
        end = min(start + win_len, len(filt))
        segment = filt[start:end]

        if len(segment) < max(10, min_distance):
            continue

        # energy-based gating
        seg_std = float(np.std(segment))
        if seg_std < min_std_threshold:
            continue

        peaks, props = find_peaks(
            segment,
            distance=max(1, min_distance),
            prominence=peak_prominence,
        )

        duration = len(segment) / fs
        peak_rate_hz = len(peaks) / max(duration, 1e-9)

        if peak_rate_hz > min_peak_rate_hz:
            total_steps += len(peaks)
            kept_windows.append((start, end, len(peaks), peak_rate_hz, seg_std))

    # global cleanup for tiny false positives
    if total_steps < min_final_steps_to_keep:
        total_steps = 0

    return {
        "steps_hat": int(total_steps),
        "filtered_signal": filt,
        "fs": fs,
        "kept_windows": kept_windows,
    }

# Path Classification Section

In [4]:
def clean_altitude(values, timestamps, max_jump=5.0, smooth_window=5):
    mask = np.ones(len(values), dtype=bool)
    last_valid = values[0]
    for i in range(1, len(values)):
        if abs(values[i] - last_valid) > max_jump:
            mask[i] = False
        else:
            last_valid = values[i]
    return median_filter(values[mask], size=smooth_window), timestamps[mask]

def extract_sensor_features(values, timestamps, prefix, window_sec=50):
    slope, _ = np.polyfit(timestamps, values, 1)
    mid = len(values) // 2
    slope_first,  _ = np.polyfit(timestamps[:mid], values[:mid], 1)
    slope_second, _ = np.polyfit(timestamps[mid:], values[mid:], 1)
    win = max(3, len(values) // 10)
    local_slopes = [
        abs(np.polyfit(timestamps[i:i+win], values[i:i+win], 1)[0])
        for i in range(len(values) - win)
    ]
    min_local_slope = min(local_slopes)
    flattest_pos = np.argmin(local_slopes) / len(local_slopes)
    t0, t_end = timestamps[0], timestamps[-1]
    first_mask = timestamps <= t0 + window_sec
    last_mask  = timestamps >= t_end - window_sec
    first_max = values[first_mask].max() if first_mask.any() else values[0]
    last_min  = values[last_mask].min()  if last_mask.any()  else values[-1]
    return {
        f'{prefix}_start':           values[0],
        f'{prefix}_end':             values[-1],
        f'{prefix}_delta':           values[-1] - values[0],
        f'{prefix}_range':           values.max() - values.min(),
        f'{prefix}_mean':            values.mean(),
        f'{prefix}_slope':           slope,
        f'{prefix}_slope_first':     slope_first,
        f'{prefix}_slope_second':    slope_second,
        f'{prefix}_slope_diff':      slope_second - slope_first,
        f'{prefix}_min_local_slope': min_local_slope,
        f'{prefix}_flattest_pos':    flattest_pos,
        f'{prefix}_window_delta':    last_min - first_max,
    }

def extract_stop_features(rec, stop_std_threshold=0.04, min_stop_sec=3.0):
    ax = rec.data['ax'].values
    ay = rec.data['ay'].values
    az = rec.data['az'].values
    ts = rec.data['ax'].timestamps
    fs = rec.data['ax'].samplerate
    a_mag = np.sqrt(ax**2 + ay**2 + az**2)
    win = max(3, int(fs * 1.0))
    n = len(a_mag)
    rolling_std = np.array([a_mag[max(0, i-win//2): i+win//2+1].std() for i in range(n)])
    is_stopped = rolling_std < stop_std_threshold
    dt = np.diff(ts, prepend=ts[0])
    n_stops, total_stop_dur, longest_stop = 0, 0.0, 0.0
    in_stop, current_dur = False, 0.0
    for i, stopped in enumerate(is_stopped):
        if stopped:
            current_dur += dt[i]; in_stop = True
        else:
            if in_stop and current_dur >= min_stop_sec:
                n_stops += 1; total_stop_dur += current_dur
                longest_stop = max(longest_stop, current_dur)
            in_stop = False; current_dur = 0.0
    if in_stop and current_dur >= min_stop_sec:
        n_stops += 1; total_stop_dur += current_dur
        longest_stop = max(longest_stop, current_dur)
    return {'n_stops': n_stops, 'total_stop_duration': total_stop_dur, 'longest_stop': longest_stop}

def _circ_mean_std(angles_deg):
    rad = np.deg2rad(angles_deg)
    S, C = np.mean(np.sin(rad)), np.mean(np.cos(rad))
    R = np.sqrt(S**2 + C**2)
    mean_dir = np.rad2deg(np.arctan2(S, C)) % 360
    circ_std  = np.rad2deg(np.sqrt(max(0.0, -2 * np.log(R + 1e-10))))
    return mean_dir, circ_std

def _circ_diff(a, b):
    return (a - b + 180) % 360 - 180

def extract_orientation_features(rec, n_segs=8):
    seg_keys = [f'azimuth_s{i}' for i in range(n_segs)]
    diff_keys = [f'azimuth_s{i}_s{i-1}' for i in range(1, n_segs)]
    NAN_KEYS = ['azimuth_mean', 'azimuth_std',
                'azimuth_first_mean', 'azimuth_mid_mean', 'azimuth_last_mean',
                'azimuth_mid_std', 'azimuth_n_turns'] + seg_keys + diff_keys

    azimuth, fs = None, 1.0

    if 'phone_orientationx' in rec.data:
        trace = rec.data['phone_orientationx']
        azimuth = trace.values
        fs = trace.samplerate
        src = 1
    elif 'mx' in rec.data and 'my' in rec.data:
        mx = rec.data['mx'].values
        my = rec.data['my'].values
        azimuth = np.rad2deg(np.arctan2(my, mx)) % 360
        fs = rec.data['mx'].samplerate
        src = 2
    else:
        return {'has_azimuth': 0, **{k: np.nan for k in NAN_KEYS}}

    win = max(3, int(fs * 5))
    s_smooth = uniform_filter1d(np.sin(np.deg2rad(azimuth)), size=win)
    c_smooth = uniform_filter1d(np.cos(np.deg2rad(azimuth)), size=win)
    az_smooth = np.rad2deg(np.arctan2(s_smooth, c_smooth)) % 360

    n = len(az_smooth)
    t1, t2 = n // 3, 2 * n // 3

    mean_all, std_all = _circ_mean_std(az_smooth)
    mean_fst, _       = _circ_mean_std(az_smooth[:t1])
    mean_mid, std_mid = _circ_mean_std(az_smooth[t1:t2])
    mean_lst, _       = _circ_mean_std(az_smooth[t2:])

    edges = [int(n * i / n_segs) for i in range(n_segs + 1)]
    seg_means = [_circ_mean_std(az_smooth[edges[i]:edges[i+1]])[0] for i in range(n_segs)]
    seg_feats = {f'azimuth_s{i}': seg_means[i] for i in range(n_segs)}
    diff_feats = {f'azimuth_s{i}_s{i-1}': _circ_diff(seg_means[i], seg_means[i-1])
                  for i in range(1, n_segs)}

    d_az = np.diff(az_smooth)
    d_az = (d_az + 180) % 360 - 180
    in_turn, n_turns = False, 0
    for da in d_az:
        if abs(da) > 10 and not in_turn:
            in_turn = True; n_turns += 1
        elif abs(da) <= 10:
            in_turn = False

    return {
        'has_azimuth':        src,
        'azimuth_mean':       mean_all,
        'azimuth_std':        std_all,
        'azimuth_first_mean': mean_fst,
        'azimuth_mid_mean':   mean_mid,
        'azimuth_last_mean':  mean_lst,
        'azimuth_mid_std':    std_mid,
        'azimuth_n_turns':    n_turns,
        **seg_feats,
        **diff_feats,
    }

def extract_path_features(rec):
    features = {}
    features['duration'] = rec.data['ax'].total_time

    pressure_keys = ['pressure_start','pressure_end','pressure_delta','pressure_range',
                     'pressure_mean','pressure_slope','pressure_slope_first',
                     'pressure_slope_second','pressure_slope_diff',
                     'pressure_min_local_slope','pressure_flattest_pos','pressure_window_delta']
    altitude_keys = [k.replace('pressure', 'altitude') for k in pressure_keys]

    if 'phone_pressure' in rec.data:
        p = rec.data['phone_pressure']
        features.update(extract_sensor_features(p.values, p.timestamps, 'pressure'))
        features['has_pressure'] = 1
    else:
        for k in pressure_keys: features[k] = np.nan
        features['has_pressure'] = 0

    if 'altitude' in rec.data:
        a = rec.data['altitude']
        cleaned, ts_clean = clean_altitude(a.values, a.timestamps)
        if len(cleaned) > 1:
            features.update(extract_sensor_features(cleaned, ts_clean, 'altitude'))
        features['has_altitude'] = 1
    else:
        for k in altitude_keys: features[k] = np.nan
        features['has_altitude'] = 0

    features.update(extract_stop_features(rec))
    features.update(extract_orientation_features(rec))

    return features

# Watch Location Section

In [ ]:
WATCH_KEYS = ("ax", "ay", "az", "gx", "gy", "gz", "mx", "my", "mz", "temperature", "altitude")
AXIS_GROUPS = {
    "acc": ("ax", "ay", "az"),
    "gyr": ("gx", "gy", "gz"),
    "mag": ("mx", "my", "mz"),
}


def log(message: str) -> None:
    timestamp = time.strftime("%H:%M:%S")
    line = f"[{timestamp}] {message}"
    print(line, flush=True)



def parse_trace_id(path: Path) -> int:
    match = re.search(r"(\d{3})\.pkl$", path.name)
    if match is None:
        raise ValueError(f"Could not parse trace id from {path.name}")
    return int(match.group(1))


def downsample(values: Iterable[float], max_len: int = 3000) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) <= max_len:
        return arr
    idx = np.linspace(0, len(arr) - 1, max_len).astype(int)
    return arr[idx]


def signal_features(values: np.ndarray, samplerate: float) -> dict[str, float]:
    names = (
        "mean",
        "std",
        "median",
        "q05",
        "q25",
        "q75",
        "q95",
        "min",
        "max",
        "range",
        "rms",
        "mad",
        "zcr",
        "dom_freq",
        "dom_power",
        "spec_entropy",
    )
    if len(values) == 0:
        return {name: 0.0 for name in names}

    mean = float(np.mean(values))
    feats = {
        "mean": mean,
        "std": float(np.std(values)),
        "median": float(np.median(values)),
        "q05": float(np.quantile(values, 0.05)),
        "q25": float(np.quantile(values, 0.25)),
        "q75": float(np.quantile(values, 0.75)),
        "q95": float(np.quantile(values, 0.95)),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
        "rms": float(np.sqrt(np.mean(values**2))),
        "mad": float(np.mean(np.abs(values - mean))),
        "zcr": float(np.mean(np.diff(np.signbit(values)) != 0)) if len(values) > 1 else 0.0,
    }
    feats["range"] = feats["max"] - feats["min"]

    if len(values) > 32 and samplerate > 0:
        freqs, power = signal.welch(values - mean, fs=samplerate, nperseg=min(256, len(values)))
        power = np.maximum(power, 1e-12)
        feats["dom_freq"] = float(freqs[np.argmax(power[1:]) + 1] if len(power) > 1 else 0.0)
        feats["dom_power"] = float(np.max(power))
        power_share = power / np.sum(power)
        feats["spec_entropy"] = float(-(power_share * np.log(power_share)).sum())
    else:
        feats["dom_freq"] = 0.0
        feats["dom_power"] = 0.0
        feats["spec_entropy"] = 0.0

    return feats


def safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    if len(a) < 4 or len(b) < 4:
        return 0.0
    corr = np.corrcoef(a, b)[0, 1]
    return 0.0 if not np.isfinite(corr) else float(corr)


def extract_watch_location_features(recording: Recording) -> dict[str, float]:
    features: dict[str, float] = {}
    if "ax" in recording.data:
        features["duration_s"] = float(recording.data["ax"].total_time)

    grouped_axes: dict[str, list[np.ndarray]] = {name: [] for name in AXIS_GROUPS}

    for key in WATCH_KEYS:
        if key not in recording.data:
            continue

        trace = recording.data[key]
        values = downsample(trace.values)
        for name, value in signal_features(values, trace.samplerate).items():
            features[f"{key}_{name}"] = value
        features[f"{key}_samplerate"] = float(trace.samplerate)
        features[f"{key}_gap"] = float(trace.max_update_gap)

        for prefix, axis_names in AXIS_GROUPS.items():
            if key in axis_names:
                max_len = 4000 if prefix != "mag" else 1500
                grouped_axes[prefix].append(downsample(trace.values, max_len=max_len))

    for prefix, axis_names in AXIS_GROUPS.items():
        axes = grouped_axes[prefix]
        if len(axes) != 3:
            continue

        usable_len = min(len(axis) for axis in axes)
        stacked = np.vstack([axis[:usable_len] for axis in axes])
        magnitude = np.sqrt((stacked**2).sum(axis=0))
        samplerate = float(np.mean([recording.data[name].samplerate for name in axis_names if name in recording.data]))

        for name, value in signal_features(downsample(magnitude), samplerate).items():
            features[f"{prefix}mag_{name}"] = value

        features[f"{prefix}_corr_xy"] = safe_corr(stacked[0], stacked[1])
        features[f"{prefix}_corr_xz"] = safe_corr(stacked[0], stacked[2])
        features[f"{prefix}_corr_yz"] = safe_corr(stacked[1], stacked[2])

    return features


def load_dataset(data_dir: Path, labeled: bool) -> tuple[pd.DataFrame, np.ndarray, np.ndarray | None, np.ndarray | None]:
    rows: list[dict[str, float]] = []
    ids: list[int] = []
    labels: list[int] = []
    groups: list[int] = []
    paths = sorted(data_dir.glob("*.pkl"))
    total = len(paths)
    start_time = time.time()

    for idx, path in enumerate(paths, start=1):
        recording = Recording(str(path))
        rows.append(extract_watch_location_features(recording))
        ids.append(parse_trace_id(path))
        if labeled:
            labels.append(int(recording.labels["watch_loc"]))
            groups.append(int(recording.labels["path_idx"]))
        if idx == 1 or idx % 25 == 0 or idx == total:
            elapsed = time.time() - start_time
            log(f"  Processed {idx}/{total} traces from {data_dir.name} in {elapsed:.1f}s")

    frame = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan)
    result = (
        frame,
        np.asarray(ids, dtype=int),
        np.asarray(labels, dtype=int) if labeled else None,
        np.asarray(groups, dtype=int) if labeled else None,
    )
    return result


def build_model() -> Pipeline:
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "classifier",
                ExtraTreesClassifier(
                    n_estimators=500,
                    max_depth=12,
                    min_samples_leaf=3,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=1,
                ),
            ),
        ]
    )


def align_feature_columns(features: pd.DataFrame, feature_columns: list[str]) -> pd.DataFrame:
    aligned = features.reindex(columns=feature_columns, fill_value=np.nan)
    return aligned


def evaluate_model(features: pd.DataFrame, labels: np.ndarray, path_groups: np.ndarray) -> None:
    log("Evaluating watch-location model...")
    model = build_model()

    train_scores: list[float] = []
    stratified_scores: list[float] = []
    stratified_confusion = np.zeros((3, 3), dtype=int)
    stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for train_idx, valid_idx in stratified_cv.split(features, labels):
        model.fit(features.iloc[train_idx], labels[train_idx])
        train_predictions = model.predict(features.iloc[train_idx])
        predictions = model.predict(features.iloc[valid_idx])
        train_scores.append(accuracy_score(labels[train_idx], train_predictions))
        stratified_scores.append(accuracy_score(labels[valid_idx], predictions))
        stratified_confusion += confusion_matrix(labels[valid_idx], predictions, labels=[0, 1, 2])

    grouped_scores: list[float] = []
    grouped_cv = GroupKFold(n_splits=5)
    for train_idx, valid_idx in grouped_cv.split(features, labels, groups=path_groups):
        model.fit(features.iloc[train_idx], labels[train_idx])
        predictions = model.predict(features.iloc[valid_idx])
        grouped_scores.append(accuracy_score(labels[valid_idx], predictions))

    model.fit(features, labels)
    final_train_predictions = model.predict(features)
    full_train_score = accuracy_score(labels, final_train_predictions)

    class_counts = pd.Series(labels).value_counts().sort_index().to_dict()
    per_class_recall = stratified_confusion.diagonal() / np.maximum(stratified_confusion.sum(axis=1), 1)

    log("Watch-location summary")
    log(f"  Training traces: {len(features)}")
    log(f"  Feature columns: {features.shape[1]}")
    log(f"  Class counts (0=wrist, 1=belt, 2=ankle): {class_counts}")
    log(f"  Full-train accuracy: {full_train_score:.4f}")
    log(f"  Mean fold training accuracy: {np.mean(train_scores):.4f} +/- {np.std(train_scores):.4f}")
    log(f"  Mean 5-fold validation accuracy: {np.mean(stratified_scores):.4f} +/- {np.std(stratified_scores):.4f}")
    log(f"  Mean 5-fold grouped-by-path validation accuracy: {np.mean(grouped_scores):.4f} +/- {np.std(grouped_scores):.4f}")
    log("  Stratified CV confusion matrix (rows=true, cols=pred):")
    log(str(stratified_confusion))
    log(
        "  Stratified CV per-class recall "
        f"(wrist, belt, ankle): {[round(float(x), 4) for x in per_class_recall]}"
    )


def train_watch_location_model(
    train_dir: Path,
    evaluate: bool = True,
    model_output: Path | None = None,
) -> dict[str, object]:
    log(f"Loading training traces from {train_dir} ...")
    train_x, _, train_y, train_groups = load_dataset(train_dir, labeled=True)

    if evaluate and train_y is not None and train_groups is not None:
        evaluate_model(train_x, train_y, train_groups)

    model = build_model()
    log("Training final model on all training traces...")
    model.fit(train_x, train_y)

    artifact = {
        "model": model,
        "feature_columns": list(train_x.columns),
        "label_map": {0: "wrist", 1: "belt", 2: "ankle"},
        "task": "watch_location",
    }

    if model_output is not None:
        model_output.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(artifact, model_output)
        log(f"Saved watch-location model to {model_output}")

    return artifact


def load_watch_location_model(model_path: Path) -> dict[str, object]:
    artifact = joblib.load(model_path)
    required_keys = {"model", "feature_columns", "task"}
    missing_keys = required_keys.difference(artifact.keys())
    if missing_keys:
        raise ValueError(f"Saved model at {model_path} is missing keys: {sorted(missing_keys)}")
    if artifact["task"] != "watch_location":
        raise ValueError(f"Saved model at {model_path} is not a watch-location model")
    log(f"Loaded watch-location model from {model_path}")
    return artifact


def predict_watch_locations_from_artifact(
    artifact: dict[str, object],
    test_dir: Path,
    output_csv: Path,
) -> pd.DataFrame:
    log(f"Loading test traces from {test_dir} ...")
    test_x, test_ids, _, _ = load_dataset(test_dir, labeled=False)
    feature_columns = artifact["feature_columns"]
    test_x = align_feature_columns(test_x, feature_columns)

    log("Predicting watch locations for test traces...")
    predicted_watch_locations = artifact["model"].predict(test_x).astype(int)

    predictions = pd.DataFrame(
        {
            "Id": test_ids,
            "watch_loc": predicted_watch_locations,
        }
    ).sort_values("Id")

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    predictions.to_csv(output_csv, index=False)
    log(f"Saved watch-location predictions to {output_csv}")
    return predictions


def predict_watch_locations(
    train_dir: Path,
    test_dir: Path,
    output_csv: Path,
    evaluate: bool,
    model_output: Path | None = None,
) -> None:
    artifact = train_watch_location_model(
        train_dir=train_dir,
        evaluate=evaluate,
        model_output=model_output,
    )
    predict_watch_locations_from_artifact(
        artifact=artifact,
        test_dir=test_dir,
        output_csv=output_csv,
    )

# Activity Detection Section


In [ ]:
ACTIVITY_ORDER = ("standing", "walking", "running", "cycling")
WATCH_MOTION_GROUPS = {
    "watch_acc": ("ax", "ay", "az"),
    "watch_gyr": ("gx", "gy", "gz"),
    "watch_mag": ("mx", "my", "mz"),
}

DEFAULT_WINDOW_S = 15.0
DEFAULT_HOP_S = 5.0
DEFAULT_MIN_ACTIVITY_S = 60.0


def normalize_activities(raw_value) -> list[str]:
    if isinstance(raw_value, Mapping):
        return [name for name in ACTIVITY_ORDER if bool(raw_value.get(name, False))]

    if isinstance(raw_value, (list, tuple, set, np.ndarray)):
        parsed = []
        for item in raw_value:
            if isinstance(item, str):
                label = item.strip().lower()
                if label in ACTIVITY_ORDER:
                    parsed.append(label)
            elif isinstance(item, (int, np.integer)) and 0 <= int(item) < len(ACTIVITY_ORDER):
                parsed.append(ACTIVITY_ORDER[int(item)])
        return [name for name in ACTIVITY_ORDER if name in set(parsed)]

    return []


def summarize_signal(values: Sequence[float], samplerate: float) -> dict[str, float]:
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        raise ValueError("Signal summary requires at least one sample.")

    centered = values - np.mean(values)
    freqs, power = welch(centered, fs=samplerate, nperseg=min(256, len(values)))
    dom_idx = np.argmax(power[1:]) + 1 if len(power) > 1 else 0

    return {
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "rms": float(np.sqrt(np.mean(values ** 2))),
        "iqr": float(np.quantile(values, 0.75) - np.quantile(values, 0.25)),
        "p10": float(np.quantile(values, 0.10)),
        "p90": float(np.quantile(values, 0.90)),
        "dom_freq": float(freqs[dom_idx]) if len(freqs) else 0.0,
        "dom_power": float(power[dom_idx]) if len(power) else 0.0,
        "spec_energy": float(power.sum()) if len(power) else 0.0,
    }


def extract_window_feature_rows(
    recording: Recording,
    motion_groups: Mapping[str, Sequence[str]] = WATCH_MOTION_GROUPS,
    window_s: float = DEFAULT_WINDOW_S,
    hop_s: float = DEFAULT_HOP_S,
) -> pd.DataFrame:
    base_keys = ("ax", "ay", "az")
    if not all(key in recording.data for key in base_keys):
        return pd.DataFrame()

    base_fs = float(np.mean([recording.data[key].samplerate for key in base_keys]))
    base_len = min(len(recording.data[key].values) for key in base_keys)
    window_n = max(int(window_s * base_fs), 1)
    hop_n = max(int(hop_s * base_fs), 1)

    if base_len < window_n:
        return pd.DataFrame()

    rows: list[dict[str, float]] = []
    for start_idx in range(0, base_len - window_n + 1, hop_n):
        end_idx = start_idx + window_n
        row = {
            "start_s": start_idx / base_fs,
            "end_s": end_idx / base_fs,
            "mid_s": (start_idx + end_idx) / (2 * base_fs),
            "window_s": float(window_s),
            "hop_s": float(hop_s),
        }

        for prefix, keys in motion_groups.items():
            if not all(key in recording.data for key in keys):
                continue

            usable_len = min(len(recording.data[key].values) for key in keys)
            if end_idx > usable_len:
                continue

            stacked = np.vstack(
                [recording.data[key].values[start_idx:end_idx].astype(float) for key in keys]
            )
            magnitude = np.sqrt((stacked ** 2).sum(axis=0))
            samplerate = float(np.mean([recording.data[key].samplerate for key in keys]))

            for feature_name, value in summarize_signal(magnitude, samplerate).items():
                row[f"{prefix}_{feature_name}"] = value

        rows.append(row)

    return pd.DataFrame(rows)


def _bridge_short_gaps(mask: Sequence[bool], max_gap_windows: int) -> np.ndarray:
    mask = np.asarray(mask, dtype=bool).copy()
    if max_gap_windows <= 0:
        return mask

    n_samples = len(mask)
    idx = 0
    while idx < n_samples:
        if mask[idx]:
            idx += 1
            continue

        gap_end = idx
        while gap_end < n_samples and not mask[gap_end]:
            gap_end += 1

        if idx > 0 and gap_end < n_samples and (gap_end - idx) <= max_gap_windows:
            mask[idx:gap_end] = True

        idx = gap_end

    return mask


def _smooth_labels(labels: Sequence[str]) -> np.ndarray:
    labels = np.asarray(labels, dtype=object)
    if len(labels) < 3:
        return labels.copy()

    smoothed = labels.copy()
    for idx in range(1, len(labels) - 1):
        triad = labels[idx - 1 : idx + 2]
        values, counts = np.unique(triad, return_counts=True)
        smoothed[idx] = values[np.argmax(counts)]
    return smoothed


def _longest_positive_run(mask: Sequence[bool], hop_s: float, window_s: float) -> float:
    best = 0.0
    current = 0
    for flag in mask:
        if flag:
            current += 1
            best = max(best, window_s + max(current - 1, 0) * hop_s)
        else:
            current = 0
    return best


def _activity_gap_windows(activity: str, hop_s: float) -> int:
    if activity in {"walking", "running"}:
        return int(np.floor(8.0 / hop_s))
    return 1


@dataclass
class ActivityRecognizer:
    classifier: ExtraTreesClassifier
    feature_columns: list[str]
    feature_medians: pd.Series
    motion_groups: Mapping[str, Sequence[str]] = field(default_factory=lambda: WATCH_MOTION_GROUPS)
    window_s: float = DEFAULT_WINDOW_S
    hop_s: float = DEFAULT_HOP_S
    min_activity_s: float = DEFAULT_MIN_ACTIVITY_S

    def label_windows(self, recording: Recording, watch_loc: int | None = None) -> pd.DataFrame:
        window_df = extract_window_feature_rows(
            recording,
            motion_groups=self.motion_groups,
            window_s=self.window_s,
            hop_s=self.hop_s,
        )
        if window_df.empty:
            return window_df
        if watch_loc is not None and not window_df.empty:
            window_df["watch_loc"] = int(watch_loc)

        x = window_df.reindex(columns=self.feature_columns).fillna(self.feature_medians)
        proba = self.classifier.predict_proba(x)
        classes = np.asarray(self.classifier.classes_)
        raw_labels = classes[np.argmax(proba, axis=1)]
        pred_labels = _smooth_labels(raw_labels)

        labeled = window_df.copy()
        labeled["raw_pred_label"] = raw_labels
        labeled["pred_label"] = pred_labels

        for activity_name in ACTIVITY_ORDER:
            labeled[f"prob_{activity_name}"] = 0.0

        for class_idx, class_name in enumerate(classes):
            labeled[f"prob_{class_name}"] = proba[:, class_idx]

        return labeled

    def summarize_labeled_windows(self, labeled_window_df: pd.DataFrame) -> pd.DataFrame:
        if labeled_window_df.empty:
            return pd.DataFrame(
                {
                    "activity": list(ACTIVITY_ORDER),
                    "longest_run_s": [0.0] * len(ACTIVITY_ORDER),
                    "meets_60s_rule": [False] * len(ACTIVITY_ORDER),
                }
            )

        hop_s = float(labeled_window_df["hop_s"].iloc[0])
        window_s = float(labeled_window_df["window_s"].iloc[0])

        rows = []
        for activity_name in ACTIVITY_ORDER:
            raw_mask = labeled_window_df["pred_label"].to_numpy() == activity_name
            bridged_mask = _bridge_short_gaps(
                raw_mask,
                max_gap_windows=_activity_gap_windows(activity_name, hop_s),
            )
            longest_run_s = _longest_positive_run(bridged_mask, hop_s=hop_s, window_s=window_s)
            rows.append(
                {
                    "activity": activity_name,
                    "longest_run_s": float(longest_run_s),
                    "meets_60s_rule": bool(longest_run_s >= self.min_activity_s),
                }
            )

        return pd.DataFrame(rows)

    def predict_activities(self, recording: Recording, watch_loc: int | None = None) -> dict[str, bool]:
        summary_df = self.summarize_labeled_windows(
            self.label_windows(recording, watch_loc=watch_loc)
        )
        return {
            row.activity: bool(row.meets_60s_rule)
            for row in summary_df.itertuples(index=False)
        }


def _trace_id_from_path(path: Path) -> int:
    return int(path.stem.split("_")[-1])


def build_activity_recognizer(
    train_dir: Path | str,
    motion_groups: Mapping[str, Sequence[str]] = WATCH_MOTION_GROUPS,
    window_s: float = DEFAULT_WINDOW_S,
    hop_s: float = DEFAULT_HOP_S,
    min_activity_s: float = DEFAULT_MIN_ACTIVITY_S,
    standing_fraction: float = 0.15,
    random_state: int = 42,
    n_estimators: int = 120,
    min_samples_leaf: int = 2,
) -> ActivityRecognizer:
    train_dir = Path(train_dir)
    metadata_rows = []
    window_cache: dict[str, pd.DataFrame] = {}
    all_feature_names: set[str] = set()

    for path in sorted(train_dir.glob("*.pkl")):
        recording = Recording(str(path))
        activities = normalize_activities((recording.labels or {}).get("activities", []))
        metadata_rows.append(
            {
                "file": path.name,
                "trace_id": _trace_id_from_path(path),
                "n_activities": len(activities),
                **{activity: activity in activities for activity in ACTIVITY_ORDER},
            }
        )

        window_df = extract_window_feature_rows(
            recording,
            motion_groups=motion_groups,
            window_s=window_s,
            hop_s=hop_s,
        )
        if not window_df.empty:
            window_df["watch_loc"] = int(recording.labels["watch_loc"])
        window_cache[path.name] = window_df
        all_feature_names.update(
            column
            for column in window_df.columns
            if column not in {"start_s", "end_s", "mid_s", "window_s", "hop_s"}
        )

    metadata_df = pd.DataFrame(metadata_rows).sort_values("trace_id").reset_index(drop=True)
    single_label_df = metadata_df.loc[metadata_df["n_activities"] == 1].copy()
    single_label_df["activity_label"] = single_label_df[list(ACTIVITY_ORDER)].idxmax(axis=1)

    training_parts = []
    for row in single_label_df.itertuples(index=False):
        window_df = window_cache[row.file]
        if window_df.empty:
            continue
        part = window_df.copy()
        part["file"] = row.file
        part["activity_label"] = row.activity_label
        training_parts.append(part)

    standing_subset = metadata_df.loc[metadata_df["standing"]].copy()
    for row in standing_subset.itertuples(index=False):
        window_df = window_cache[row.file]
        if window_df.empty:
            continue

        motion_score = window_df["watch_acc_std"] + 0.3 * window_df["watch_gyr_std"]
        keep_n = max(1, int(np.ceil(standing_fraction * len(window_df))))
        part = window_df.loc[motion_score.nsmallest(keep_n).index].copy()
        part["file"] = row.file
        part["activity_label"] = "standing"
        training_parts.append(part)

    if not training_parts:
        raise RuntimeError("No training windows were extracted for activity recognition.")

    training_df = pd.concat(training_parts, ignore_index=True)
    feature_columns = sorted(all_feature_names)
    feature_medians = training_df.reindex(columns=feature_columns).median(numeric_only=True)
    x_train = training_df.reindex(columns=feature_columns).fillna(feature_medians)
    y_train = training_df["activity_label"]

    classifier = ExtraTreesClassifier(
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        class_weight="balanced",
        random_state=random_state,
        n_jobs=1,
    )
    classifier.fit(x_train, y_train)

    return ActivityRecognizer(
        classifier=classifier,
        feature_columns=feature_columns,
        feature_medians=feature_medians,
        motion_groups=motion_groups,
        window_s=window_s,
        hop_s=hop_s,
        min_activity_s=min_activity_s,
    )


def evaluate_activity_recognizer(
    recognizer: ActivityRecognizer,
    data_dir: Path | str,
) -> pd.DataFrame:
    data_dir = Path(data_dir)
    rows = []
    for path in sorted(data_dir.glob("*.pkl")):
        recording = Recording(str(path))
        truth_activities = normalize_activities((recording.labels or {}).get("activities", []))
        truth = {activity: activity in truth_activities for activity in ACTIVITY_ORDER}
        pred = recognizer.predict_activities(recording)

        row = {"file": path.name}
        for activity in ACTIVITY_ORDER:
            row[f"true_{activity}"] = bool(truth[activity])
            row[f"pred_{activity}"] = bool(pred[activity])
        rows.append(row)

    return pd.DataFrame(rows)


def activity_f1_summary(prediction_df: pd.DataFrame) -> pd.Series:
    scores = {
        activity: f1_score(
            prediction_df[f"true_{activity}"],
            prediction_df[f"pred_{activity}"],
        )
        for activity in ACTIVITY_ORDER
    }
    scores["macro"] = float(np.mean([scores[activity] for activity in ACTIVITY_ORDER]))
    return pd.Series(scores, dtype=float)

# Final Prediction Section

In [ ]:
filenames = sorted(TEST_DIR.glob("*.pkl"))


### Activity Recognition

# Train the activity recognizer directly in this notebook
activity_recognizer = build_activity_recognizer(TRAIN_DIR)

#Save/output the trained model as a .joblib file.
activity_model_path = Path('group10_model_activity_watchloc.joblib')
joblib.dump(activity_recognizer, activity_model_path)

In [ ]:
### Smartwatch Location

# Train the watch-location model
train_dir = Path('/kaggle/input/competitions/26-mham-ex3/data/train')

watch_location_artifact = train_watch_location_model(
    train_dir=train_dir,
    evaluate=True,
    model_output=None,
)

#Save the trained model artifact as a .joblib file.
joblib.dump(watch_location_artifact, 'group10_model_watchloc.joblib')

In [ ]:

###Path

all_features, all_labels = [], []
for f in filenames:
    r = Recording(str(f))
    all_labels.append(r.labels['path_idx'])
    all_features.append(extract_path_features(r))
X = pd.DataFrame(all_features)
labels = np.array(all_labels)
X['label'] = labels
X.pop('label')

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=63,
        random_state=42,
        verbose=-1,
    )),
])

# Train
model = pipeline.fit(X, labels)

# Save model
path_model = {
    "model": model,
    "feature_columns": list(X.columns),
    "task": "path_idx",
}

joblib.dump(path_model, "group10_model_path.joblib")
print("Saved group10_model_path.joblib")


In [ ]:
submission = []

for path in filenames:
    recording = Recording(str(path))
    id = parse_trace_id(path)

    step_pred = int(estimate_steps_windowed(recording)["steps_hat"])
    
    watch_loc_features = pd.DataFrame([extract_watch_location_features(recording)])
    watch_loc_features = align_feature_columns(
        watch_loc_features,
        watch_location_artifact["feature_columns"],
    )
    watch_loc = int(watch_location_artifact["model"].predict(watch_loc_features)[0])

    predicted_activities = activity_recognizer.predict_activities(
        recording,
        watch_loc=watch_loc,
    )

    path_features = pd.DataFrame([extract_path_features(recording)])
    path_idx = int(path_model.predict(path_features)[0])

    predicted_activities = activity_recognizer.predict_activities(
        recording,
        watch_loc=watch_loc,
    )
    

    predictions = {
        "Id": id,
        "watch_loc": watch_loc,
        "path_idx": path_idx,
        "standing": bool(predicted_activities["standing"]),
        "walking": bool(predicted_activities["walking"]),
        "running": bool(predicted_activities["running"]),
        "cycling": bool(predicted_activities["cycling"]),
        "step_count": step_pred,
    }

    submission.append(predictions)

In [ ]:
# Write the predicted values into a .csv file to then upload the .csv file to Kaggle
# When cross-checking the .csv file on your computer, we recommend using a text editor and NOT excel so that the results are displayed correctly
# IMPORTANT: Do NOT change the name of the columns of the .csv file ("Id", "watch_loc", "path_idx", "standing", "walking", "running", "cycling", "step_count")
submission_df = pd.DataFrame(submission, columns=['Id', 'watch_loc', 'path_idx', 'standing', 'walking', 'running', 'cycling', 'step_count'])
#submission_df.to_csv('submission.csv', index=False)  # Saves the submission file in the current working directory
submission_df.to_csv('/kaggle/working/submission.csv', index=False)